# SAM3 / YOLO11x-Pose Basketball Detection
Set `RUN_ENV` and `DETECTION_MODEL` in **cell 2** — that's the only cell you need to edit.

In [ ]:
# ── 1. Mount Drive (Colab only) ───────────────────────────────────────────────
import sys

# RUN_ENV is defined in cell 2; this cell is safe to run first on Colab.
# On local, the import is skipped.
try:
    _run_env = RUN_ENV  # noqa: F821 — defined in cell 2 if already run
except NameError:
    _run_env = "colab"  # default assumption when running top-to-bottom

if _run_env == "colab":
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("Local env — skipping Drive mount.")

In [ ]:
# ── 2. Configuration — ONLY cell you need to edit ────────────────────────────
RUN_ENV          = "local"          # "local" | "colab"
DETECTION_MODEL  = "yolo11x-pose"   # "yolo11x-pose" | "sam3"
TRACKER          = "botsort"        # "botsort" | "bytetrack"  (yolo only)
CONF             = 0.25
IMGSZ            = 1280
HALF             = True
VIDEO_STEM       = "djurgarden1"    # filename without extension

In [ ]:
# ── 3. Path resolution ────────────────────────────────────────────────────────
import sys
import torch
from pathlib import Path

if RUN_ENV == "local":
    PROJECT_DIR = Path(".")
    VIDEO_PATH  = Path("../data/videos") / f"{VIDEO_STEM}.mp4"
    OUTPUT_DIR  = Path("output")
    VIDEO_LOCAL = VIDEO_PATH          # no copy needed
else:
    PROJECT_DIR = Path("/content/drive/MyDrive/sam3-experiments")
    VIDEO_PATH  = PROJECT_DIR / "data/videos" / f"{VIDEO_STEM}.mp4"
    OUTPUT_DIR  = PROJECT_DIR / "output"
    VIDEO_LOCAL = Path("/content") / f"{VIDEO_STEM}.mp4"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Device selection
if RUN_ENV == "colab":
    device = "0"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"RUN_ENV:         {RUN_ENV}")
print(f"DETECTION_MODEL: {DETECTION_MODEL}")
print(f"VIDEO_PATH:      {VIDEO_PATH}")
print(f"OUTPUT_DIR:      {OUTPUT_DIR}")
print(f"device:          {device}")

In [ ]:
# ── 4. Install dependencies ───────────────────────────────────────────────────
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "ultralytics==8.4.21", "opencv-python-headless", "-q"],
    check=True,
)

In [ ]:
# ── 5. GPU / device check ─────────────────────────────────────────────────────
import torch

if RUN_ENV == "colab":
    import subprocess
    subprocess.run(["nvidia-smi"])
else:
    print(f"PyTorch version: {torch.__version__}")
    print(f"MPS available:   {torch.backends.mps.is_available()}")
    print(f"CUDA available:  {torch.cuda.is_available()}")
    print(f"Using device:    {device}")

In [ ]:
# ── 6. Copy video to local storage (Colab only) ───────────────────────────────
import shutil

if RUN_ENV == "colab":
    if not VIDEO_LOCAL.exists():
        print(f"Copying {VIDEO_PATH} → {VIDEO_LOCAL} ...")
        shutil.copy(str(VIDEO_PATH), str(VIDEO_LOCAL))
        print("Done.")
    else:
        print(f"Already exists: {VIDEO_LOCAL}")
else:
    print("Local env — no copy needed.")

In [ ]:
# ── 7. Run detection ──────────────────────────────────────────────────────────
import subprocess, sys

if DETECTION_MODEL == "yolo11x-pose":
    script = PROJECT_DIR / "run_yolo_pose_track.py"
    cmd = [
        sys.executable, str(script),
        str(VIDEO_LOCAL),
        "--tracker",    TRACKER,
        "--imgsz",      str(IMGSZ),
        "--conf",       str(CONF),
        "--device",     device,
        "--output-dir", str(OUTPUT_DIR),
        *(["--half"] if HALF else []),
    ]
elif DETECTION_MODEL == "sam3":
    script = PROJECT_DIR / "run_sam3.py"
    model  = PROJECT_DIR / "sam3.pt"
    cmd = [
        sys.executable, str(script),
        str(VIDEO_LOCAL),
        "--model",      str(model),
        "--imgsz",      str(IMGSZ),
        "--conf",       str(CONF),
        "--device",     device,
        "--output-dir", str(OUTPUT_DIR / "visualizations"),
        *(["--half"] if HALF else []),
    ]
else:
    raise ValueError(f"Unknown DETECTION_MODEL: {DETECTION_MODEL}")

print("Running:", " ".join(cmd), "\n")

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end="", flush=True)

process.wait()
if process.returncode != 0:
    raise subprocess.CalledProcessError(process.returncode, cmd)

In [ ]:
# ── 8. Compress + play output video ──────────────────────────────────────────
import subprocess
from IPython.display import Video

suffix = "yolo_pose" if DETECTION_MODEL == "yolo11x-pose" else "sam3"
out_path       = OUTPUT_DIR / "visualizations" / f"{VIDEO_STEM}_{suffix}.mp4"
out_compressed = OUTPUT_DIR / "visualizations" / f"{VIDEO_STEM}_{suffix}_compressed.mp4"

subprocess.run(
    ["ffmpeg", "-y", "-loglevel", "error",
     "-i", str(out_path),
     "-vcodec", "libx264", "-crf", "28",
     str(out_compressed)],
    check=True,
)
print(f"Compressed video saved to: {out_compressed}")

Video(str(out_compressed), embed=True, width=960)